# Scoring Review: Retrieval Augmented Generation (RAG)

In this notebook, we will demonstrate how reviewers can use RAG to perform their reviews by referencing appropriate external context per item.

## Setting up the notebook

High-level configs

In [1]:
%reload_ext autoreload
%autoreload 2

from dotenv import load_dotenv

# Load environment variables from .env file. Adjust the path to the .env file as needed.
load_dotenv(dotenv_path='../.env')

# Enable asyncio in Jupyter
import asyncio
import nest_asyncio

nest_asyncio.apply()

#  Add the package to the path (required if you are running this notebook from the examples folder)
import sys
sys.path.append('../../')


Import required packages

In [2]:
import json
import numpy as np
from openai import AsyncOpenAI
import pandas as pd
from pydantic import BaseModel
from tqdm.auto import tqdm

from lattereview.providers import OpenAIProvider
from lattereview.providers import LiteLLMProvider
from lattereview.agents import ScoringReviewer
from lattereview.workflows import ReviewWorkflow

## Data

Building five example stories and dummy question-answering pairs from each story:

In [3]:
class BuildStoryOutput(BaseModel):
    story: str
    questions: list[str]
    answers: list[bool]

async def build_story():
    prompt = """
    Write a one-paragraph story with whatever realistic or imaginary theme you like,  
    then create three TRUE/FALSE questions based on your story. 
    Ensure that only readers of your story can determine whether the statements are true or false. 
    Do not reveal the answers to your questions.
    Return your story, a Python list of three questions, and another Python list of three boolean responses to the questions as your output.
    """
    provider = OpenAIProvider(model="gpt-6-sol", response_format_class=BuildStoryOutput)
    return await provider.get_json_response(prompt)

def run_build_story():
    response =  asyncio.run(build_story())[0]
    return response

data = {
    "question": [],
    "answer": [],
    "story": []
}
for i in tqdm(range(5)):
    out = json.loads(run_build_story())
    for j in range(3):
        data["question"].append(out["questions"][j])
        data["answer"].append(out["answers"][j])
        data["story"].append(out["story"])


data = pd.DataFrame(data)
data.to_csv("data.csv", index=False)
data

100%|██████████| 5/5 [00:24<00:00,  4.89s/it]


,question,answer,story
0,Did Mara open the lighthouse museum every Thur...,True,"Every Thursday, Mara opened the lighthouse mus..."
1,Was the envelope hidden inside the telescope?,False,"Every Thursday, Mara opened the lighthouse mus..."
2,Did Mara discover who made the glass fox?,False,"Every Thursday, Mara opened the lighthouse mus..."
3,Did Mira find the silver envelope beneath a bl...,True,"Every Thursday at dusk, Mira opened the tiny o..."
4,Did the map lead Mira to a meteorite behind th...,False,"Every Thursday at dusk, Mira opened the tiny o..."
5,Did Mira release the fireflies beside the river?,True,"Every Thursday at dusk, Mira opened the tiny o..."
6,Inez found the brass key inside a jar of black...,True,"On the night the town’s lighthouse went dark, ..."
7,Inez rode a bicycle up the hill to the lightho...,False,"On the night the town’s lighthouse went dark, ..."
8,The key was still in Inez’s coat pocket the ne...,True,"On the night the town’s lighthouse went dark, ..."
9,Did Mara open the clock tower every Thursday?,True,"Every Thursday, Mara opened the clock tower to..."


Embedding the stories to build a vector base:

In [4]:
data = pd.read_csv("data.csv")

async def get_embedding(text):
    client = AsyncOpenAI()
    if isinstance(text, str):
        text = [text]
    text = [x.replace("\n", " ") for x in text]         
    out = await client.embeddings.create(
        model="text-embedding-3-small",
        input=text,
        encoding_format="float"
    )
    out = [np.array(x.embedding) for x in out.data]
    return out if len(out) > 1 else out[0]

stories = {story: None for story in set(data["story"].tolist())}

# Create async tasks for all embeddings
async def process_embeddings():
    tasks = [get_embedding(story) for story in stories.keys()]
    embeddings = await asyncio.gather(*tasks)
    return list(zip(embeddings, stories.keys()))

# Run the async code and get results
vector_story_pairs = await process_embeddings()
vector_base = np.array([x[0] for x in vector_story_pairs])
vector_base

array([[ 0.01705933,  0.01777649,  0.02674866, ..., -0.03466797,
        -0.01380157, -0.04055786],
       [ 0.03114319,  0.02444458, -0.00531769, ..., -0.03369141,
         0.01721191, -0.01824951],
       [ 0.01878357,  0.02049255, -0.03071594, ..., -0.02793884,
        -0.00105858,  0.00804138],
       [-0.00632477,  0.00538635,  0.02554321, ..., -0.01611328,
        -0.02919006, -0.03414917],
       [ 0.00767136,  0.02253723, -0.02088928, ..., -0.02290344,
        -0.03314209, -0.00782776]], shape=(5, 1536))

## Retrieval
Here, we will define a simple retrieval function that takes an input text (in our case, a statement) as input, embeds it, and finds the reference text (in our case, a story) with the highest cosine similarity to the embedding of that statement. The output of this function can then be used by the reviewer agent as additional context for each item it reviews.

In [5]:
async def find_relevant_story(statement):
    s_embeddings = await get_embedding(statement)
    dot_product = np.dot(vector_base, s_embeddings)
    base_norms = np.linalg.norm(vector_base, axis=1)
    query_norm = np.linalg.norm(s_embeddings)
    cosine_similarities = dot_product / (base_norms * query_norm)
    retrieved_index = np.argmax(cosine_similarities)
    retrieved_story = vector_story_pairs[retrieved_index][1]
    return retrieved_story

input_index = 11
statement = data.iloc[input_index]["question"]
retrieved_story = await find_relevant_story(statement)

print(f"=== The question was chosen from row {input_index} ===\n{statement}")
print(f"=== The related story to the question ===\n{data.iloc[input_index]['story']}")
print(f"=== The retrieved Story ===\n{retrieved_story}")

=== The question was chosen from row 11 ===
Did the librarian recognize Mara’s grandmother’s handwriting?
=== The related story to the question ===
Every Thursday, Mara opened the clock tower to wind its brass mechanism. One rainy morning, she found a blue envelope tucked behind the pendulum. Inside was a sketch of the town square with a small X beneath the oldest bench. Mara waited until the rain stopped, then lifted a loose paving stone near the bench and found a tin box full of handwritten recipes. She took the box to the library, where the librarian recognized her grandmother’s handwriting.
=== The retrieved Story ===
Every Thursday, Mara opened the clock tower to wind its brass mechanism. One rainy morning, she found a blue envelope tucked behind the pendulum. Inside was a sketch of the town square with a small X beneath the oldest bench. Mara waited until the rain stopped, then lifted a loose paving stone near the bench and found a tin box full of handwritten recipes. She took th

## Scoring with Retrieval Augmented Generation

Here, we will create a simple reviewer agent that processes each input statement. The key argument we define for this reviewer is the `additional_context` argument. Note how we pass the `find_relevant_story` function to this `additional_context` argument, enabling the reviewer to apply this function to each input item. This allows the agent to retrieve the most relevant context for each statement before conducting its review. In our case, the reviewer will decide whether the statement is true or false based on the retrieved context, which, in this case, is the corresponding story.

In [6]:
reviewer = ScoringReviewer(
    provider=LiteLLMProvider(model="gpt-6-luna"),
    name="reviewer",
    max_concurrent_requests=20, 
    backstory="A frequent book reader",
    input_description="TRUE/FALSE questions about stories",
    model_args={"reasoning_effort": "low"},
    reasoning = "brief",
    scoring_task="Decide if the input statement is True or False given the provided story in the provided context",
    scoring_set=[1, 2],
    scoring_rules='Score 1 if the statement is TRUE and 2 if the statement is FALSE.',
    additional_context = find_relevant_story
)

review = ReviewWorkflow(
    workflow_schema=[
        {
            "round": 'A',
            "reviewers": [reviewer],
            "text_inputs": ["question"]
        }
    ]
)

updated_data = asyncio.run(review(data))
updated_data


====== Starting review round A (1/1) ======

Processing 15 eligible rows


['round: A', 'reviewer_name: reviewer'] -                     2026-09-27 13:30:46: 100%|██████████| 15/15 [00:01<00:00,  8.94it/s]

Reviewer reviewer: 15 successful, 0 failed parses
The following columns are present in the dataframe at the end of reviewer's review in round A: ['question', 'answer', 'story', 'round-A_reviewer_output', 'round-A_reviewer_reasoning', 'round-A_reviewer_score', 'round-A_reviewer_certainty']


,question,answer,story,round-A_reviewer_output,round-A_reviewer_reasoning,round-A_reviewer_score,round-A_reviewer_certainty
0,Did Mara open the lighthouse museum every Thur...,True,"Every Thursday, Mara opened the lighthouse mus...",{'reasoning': 'The statement is false: Mara op...,The statement is false: Mara opened the lighth...,2,100
1,Was the envelope hidden inside the telescope?,False,"Every Thursday, Mara opened the lighthouse mus...",{'reasoning': 'The envelope was tucked beneath...,The envelope was tucked beneath the telescope’...,2,100
2,Did Mara discover who made the glass fox?,False,"Every Thursday, Mara opened the lighthouse mus...",{'reasoning': 'The story says Mara never disco...,The story says Mara never discovered who made ...,2,100
3,Did Mira find the silver envelope beneath a bl...,True,"Every Thursday at dusk, Mira opened the tiny o...",{'reasoning': 'The story says Mira found a sil...,The story says Mira found a silver envelope tu...,1,100
4,Did the map lead Mira to a meteorite behind th...,False,"Every Thursday at dusk, Mira opened the tiny o...",{'reasoning': 'The map led Mira to a jar of gl...,The map led Mira to a jar of glowing fireflies...,2,100
5,Did Mira release the fireflies beside the river?,True,"Every Thursday at dusk, Mira opened the tiny o...",{'reasoning': 'The story says Mira opened the ...,The story says Mira opened the jar beside the ...,1,100
6,Inez found the brass key inside a jar of black...,True,"On the night the town’s lighthouse went dark, ...",{'reasoning': 'The story explicitly says Inez ...,The story explicitly says Inez found a brass k...,1,100
7,Inez rode a bicycle up the hill to the lightho...,False,"On the night the town’s lighthouse went dark, ...",{'reasoning': 'The statement is false: Inez wa...,The statement is false: Inez walked up the hil...,2,100
8,The key was still in Inez’s coat pocket the ne...,True,"On the night the town’s lighthouse went dark, ...",{'reasoning': 'The story explicitly says that ...,The story explicitly says that by morning the ...,1,100
9,Did Mara open the clock tower every Thursday?,True,"Every Thursday, Mara opened the clock tower to...",{'reasoning': 'The story explicitly says Mara ...,The story explicitly says Mara opened the cloc...,1,100


Last but not least, you can check the memory of the agent for every item it has reviewed.

In [7]:
# An example review item (pass the index for a row in the input dataframe)

reviewer.memory[11]

{'system_prompt': "Your name is: <<reviewer>> Your backstory is: <<A frequent book reader>>. Your task is to review input itmes with the following description: <<TRUE/FALSE questions about stories>>. Your final output should have the following keys: reasoning (<class 'str'>), score (<class 'int'>), certainty (<class 'int'>).",
 'model_args': {'reasoning_effort': 'low'},
 'input_prompt': '**Review the input item below and complete the scoring task as instructed:** --- **Input item:** <<Review Task ID: A-11 === question === Did the librarian recognize Mara’s grandmother’s handwriting?>> **Scoring task:** <<Decide if the input statement is True or False given the provided story in the provided context>> --- **Instructions:** 1. **Score** the input item using only the values in this set: [1, 2]. 2. Follow these rules when determining your score: <<Score 1 if the statement is TRUE and 2 if the statement is FALSE.>>. 3. After assigning a score, report your certainty level as a value between 